# PyPSA-DE: wiederverwendbarer Testlauf


### Optimieren

In [0]:
from pathlib import Path
import os
import shutil
import subprocess

# 1. Repository kopieren
src = Path("/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test")
dst = Path("/local_disk0/pypsa-de-test")

if not src.is_dir():
    raise FileNotFoundError(f"Quellordner nicht gefunden: {src}")


def walk_error(error):
    raise error


for directory, dirs, files in os.walk(src, onerror=walk_error):
    directory = Path(directory)
    relative = directory.relative_to(src)

    # Diese beiden Verzeichnisse direkt im Repo überspringen.
    if directory == src:
        for name in [".pixi", "scripts_LuKa"]:
            if name in dirs:
                print(f"Übersprungen: {name}", flush=True)
                dirs.remove(name)

    target = dst / relative
    target.mkdir(parents=True, exist_ok=True)

    for name in files:
        # Vorhandene Zieldateien ersetzen.
        shutil.copyfile(directory / name, target / name)

print(f"Repo nach {dst} kopiert.", flush=True)
print(
    f"pixi.toml vorhanden: {(dst / 'pixi.toml').exists()}",
    flush=True,
)

# 2. Installation, Umgebungsprüfung und Optimierung
script = r"""
set -euxo pipefail

RUN="20261027_2"                                                            ######### DEFINE #########
YEARS="2025 2030 2035 2040 2045 2050"                                       ######### DEFINE #########

## Pixi installieren, falls noch nicht vorhanden
export PATH="$HOME/.pixi/bin:$PATH"

if ! command -v pixi >/dev/null 2>&1; then
  curl -fsSL https://pixi.sh/install.sh | sh
fi

pixi --version

## Lokales Arbeitsverzeichnis
mkdir -p /local_disk0/pypsa-de-test
cd /local_disk0/pypsa-de-test

## Projektumgebung installieren
pixi install

## Python und Paketversionen prüfen
pixi run python -c '
import sys
import numpy
import geopandas

print("Python:", sys.executable)
print("NumPy:", numpy.__version__)
print("GeoPandas:", geopandas.__version__)
'

## Snakemake-Version prüfen
pixi run snakemake --version

## Szenariokonfigurationen erzeugen
pixi run snakemake -c1 build_scenarios

## Alle Jahre nacheinander optimieren
for YEAR in $YEARS; do

  TARGET="results/${RUN}/SR27_S3/networks/base_s_27__none_${YEAR}.nc"

  echo "=========================================="
  echo "Optimierung ${YEAR}"
  echo "Target: ${TARGET}"
  echo "=========================================="

  PYTHONPATH="" pixi run snakemake -c8 --jobs 1 \
    "$TARGET" \
    --resources mem_mb=60000 \
    --set-resources solve_sector_network_myopic:mem_mb=60000

  echo "=== ${YEAR} erfolgreich abgeschlossen ==="

done
"""

# 3. Bash-Skript genau einmal ausführen.
# Ausgabe und Fehlermeldungen fortlaufend gemeinsam anzeigen.
with subprocess.Popen(
    ["bash", "-lc", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    exit_code = process.wait()

# Bei einem Fehler auch die Notebook-Zelle fehlschlagen lassen.
if exit_code != 0:
    raise RuntimeError(
        f"Installation oder Modelllauf fehlgeschlagen: Exit-Code {exit_code}"
    )

print("Alle Schritte erfolgreich abgeschlossen.", flush=True)



### Speichern

In [0]:
# 4. Ergebnisse nach erfolgreicher Optimierung ins Volume kopieren
source = Path(
    "/local_disk0/pypsa-de-test/results/20261027_2/SR27_S3/networks"                                ######### DEFINE #########
)
destination = Path(
    "/Volumes/playground/u_luisa_kamp/pypsa-de_data/"
    "SR27-3/results/modified-4"                                                                     ######### DEFINE #########
)

# Vor dem Kopieren prüfen, ob alle erwarteten Netzwerke vorliegen.
years = [2025, 2030, 2035, 2040, 2045, 2050]
missing = [
    year for year in years
    if not (source / f"base_s_27__none_{year}.nc").is_file()
]
if missing:
    raise FileNotFoundError(f"Ergebnisdateien fehlen für: {missing}")

# Alle Dateien einschließlich Unterordner kopieren.
files = sorted(p for p in source.rglob("*") if p.is_file())

for file in files:
    target = destination / file.relative_to(source)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(file, target)

    # Einfache Kontrolle: Dateigrößen müssen übereinstimmen.
    if file.stat().st_size != target.stat().st_size:
        raise IOError(f"Dateigröße stimmt nicht überein: {target}")

    print(f"Gespeichert: {target}", flush=True)

print(f"{len(files)} Dateien ins Volume kopiert.", flush=True)